# 04c — Post-LoRA Attribution

**Runs on RunPod GPU. Always start from a FRESH kernel.**

Loads the merged LoRA checkpoint (`my_work/checkpoints/lora_combined_merged/`)
produced by `04b_lora_training.ipynb` and runs attribution on **every row** of the
frozen probe set (`prompts_triangle_v2.jsonl`) — **binary and numeric** `task_type`
slices, mirroring attribution targets from **`02`**.

**Why a separate notebook?**  Training leaves ~12 GB of GPU memory that cannot
be freed without a kernel restart (`empty_cache()` only releases the PyTorch
cache, not memory held by the CUDA process).  Starting fresh gives the full
19–24 GB to Gemma-2-2B + transcoders + attribution.

**Prerequisites:**
- `04_lora_training_data.ipynb` run (probe JSONL exists)
- `04b_lora_training.ipynb` run through §7 (`lora_combined_merged/` exists)

**Outputs:**
- Pilot: `my_work/results/statistics/stats_lora_combined_pilot.json`
- Full:  `my_work/results/statistics/stats_lora_combined.json`

**Checkpointing:** Stats append after each prompt. Re-running §5 skips **`prompt_id`s** already succeeded in **`STATS_FILE`**.

**Pilot → full:** **`RUN_FULL=True`** makes §5 copy successful pilot rows into the full stats file (instant, no GPU) so attribution is never paid twice; use **`stats_lora_combined.json`** in **`05`** with **`USE_PILOT=False`** when the sweep is finished.

## 0 — Environment setup

In [1]:
import os
import sys
from pathlib import Path

# Must be set BEFORE torch is imported anywhere.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

IN_RUNPOD = os.environ.get("RUNPOD_POD_ID") is not None

def _find_repo_root():
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "circuit_tracer" / "__init__.py").is_file():
            return directory
    workspace = Path("/workspace")
    if workspace.is_dir():
        for child in workspace.iterdir():
            if child.is_dir() and (child / "circuit_tracer" / "__init__.py").is_file():
                return child
    repo_override = os.environ.get("CT_REPO_DIR")
    if repo_override:
        p = Path(repo_override).expanduser().resolve()
        if (p / "circuit_tracer" / "__init__.py").is_file():
            return p
    return None

_root = _find_repo_root()
if _root is not None:
    if str(_root) not in sys.path:
        sys.path.insert(0, str(_root))
    _my_work = _root / "my_work"
    if str(_my_work) not in sys.path:
        sys.path.insert(0, str(_my_work))
    print(f"Repo root: {_root}")
else:
    print("WARNING: could not locate circuit_tracer repo. Set CT_REPO_DIR.")

try:
    from dotenv import load_dotenv
    if _root is not None and (_root / ".env").is_file():
        load_dotenv(_root / ".env")
except ImportError:
    pass

MY_WORK = _my_work if _root else Path(".").resolve()

if IN_RUNPOD:
    persistent_root = Path(os.environ.get("RUNPOD_PERSISTENT_ROOT", "/workspace")).resolve()
    hf_home = persistent_root / "hf"
    for key, path in {
        "HF_HOME": hf_home,
        "HUGGINGFACE_HUB_CACHE": hf_home / "hub",
        "TRANSFORMERS_CACHE": hf_home / "transformers",
    }.items():
        path.mkdir(parents=True, exist_ok=True)
        os.environ[key] = str(path)

STATS_DIR  = MY_WORK / "results" / "statistics"
GRAPHS_DIR = MY_WORK / "results" / "graphs_lora_combined"
STATS_DIR.mkdir(parents=True, exist_ok=True)
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

print(f"MY_WORK  : {MY_WORK}")

Repo root: /workspace/thesis_circuit_breaker
MY_WORK  : /workspace/thesis_circuit_breaker/my_work


## 1 — Constants

In [7]:
import json
import time
import gc
import traceback
import random

import torch

MODEL_NAME      = "google/gemma-2-2b"
TRANSCODER_NAME = "gemma"
TOKEN_TRUE      = " True"
TOKEN_FALSE     = " False"
VOCAB_ID_TRUE   = 5569
VOCAB_ID_FALSE  = 7662

ATTR_KWARGS = dict(
    batch_size=256,
    max_feature_nodes=8192,
    offload="disk",
    verbose=False,
)
PHASE = "lora_combined"

# Pilot vs full
# Start RUN_FULL=False for ~N_PILOT prompts. Set RUN_FULL=True to run everyone;
# §5 skips rows already saved and copies pilot successes into the full file (no rerun).
RUN_FULL = True
N_PILOT  = 40

MERGED_DIR       = MY_WORK / "checkpoints" / "lora_combined_merged"
PROBE_JSONL      = MY_WORK / "data" / "prompts_triangle_v2.jsonl"
PILOT_STATS_FILE = STATS_DIR / "stats_lora_combined_pilot.json"
FULL_STATS_FILE  = STATS_DIR / "stats_lora_combined.json"
STATS_FILE       = FULL_STATS_FILE if RUN_FULL else PILOT_STATS_FILE

assert MERGED_DIR.exists(), (
    f"Merged checkpoint not found: {MERGED_DIR}\n"
    "Run 04b_lora_training.ipynb through §7 first."
)

device_str = "cuda" if torch.cuda.is_available() else "cpu"
device     = torch.device(device_str)

print(f"MERGED_DIR : {MERGED_DIR}")
print(f"STATS_FILE : {STATS_FILE}")
print(f"RUN_FULL   : {RUN_FULL}  (pilot N={N_PILOT})")
print(f"CUDA       : {torch.cuda.is_available()}", end="")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"  |  Free VRAM: {free/1e9:.1f} GB / {total/1e9:.1f} GB")
else:
    print()

MERGED_DIR : /workspace/thesis_circuit_breaker/my_work/checkpoints/lora_combined_merged
STATS_FILE : /workspace/thesis_circuit_breaker/my_work/results/statistics/stats_lora_combined.json
RUN_FULL   : True  (pilot N=40)
CUDA       : True  |  Free VRAM: 9.4 GB / 21.0 GB


## 2 — Load merged model into ReplacementModel

TransformerLens requires a registered `model_name`; pass the local merged
weights via `hf_model=`.  We load `merged_hf` on CPU so that only one copy
of Gemma-2-2B lives on the GPU (inside HookedTransformer) at any time.

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from circuit_tracer import ReplacementModel

print("Loading merged weights on CPU ...")
merged_hf = AutoModelForCausalLM.from_pretrained(
    str(MERGED_DIR),
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map="cpu",
).eval()

print("Building ReplacementModel (moves weights to GPU) ...")
ct_model = ReplacementModel.from_pretrained(
    MODEL_NAME,
    TRANSCODER_NAME,
    dtype=torch.bfloat16,
    backend="transformerlens",
    device=device,
    lazy_encoder=True,
    lazy_decoder=True,
    hf_model=merged_hf,
)

# Free the CPU copy immediately — HookedTransformer now holds the weights on GPU.
del merged_hf
gc.collect()

ct_tokenizer = ct_model.tokenizer

id_true  = ct_tokenizer.encode(TOKEN_TRUE,  add_special_tokens=False)[-1]
id_false = ct_tokenizer.encode(TOKEN_FALSE, add_special_tokens=False)[-1]
assert id_true  == VOCAB_ID_TRUE,  f"Token mismatch True:  {id_true}"
assert id_false == VOCAB_ID_FALSE, f"Token mismatch False: {id_false}"
print(f"ReplacementModel ready. Token IDs: True={id_true}, False={id_false}")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM after load: {(total-free)/1e9:.1f} GB used / {total/1e9:.1f} GB total")

/workspace/venvs/ct/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading merged weights on CPU ...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Building ReplacementModel (moves weights to GPU) ...


Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer
ReplacementModel ready. Token IDs: True=5569, False=7662
VRAM after load: 8.6 GB used / 21.0 GB total


## 3 — Load full probe set (binary + numeric) and optional pilot subset

In [8]:
from collections import Counter, defaultdict

def _resolve_attr_targets(entry: dict) -> list[str]:
    """Same logic as `02_baseline_attribution.ipynb` (single-token numeric targets)."""
    task_type = entry.get("task_type", "binary")
    if task_type == "binary":
        return [TOKEN_TRUE, TOKEN_FALSE]
    raw_label = entry["label_token"]
    ids_raw = ct_tokenizer.encode(raw_label, add_special_tokens=False)
    if len(ids_raw) == 1:
        return [raw_label]
    alt = raw_label.lstrip()
    ids_alt = ct_tokenizer.encode(alt, add_special_tokens=False)
    if len(ids_alt) == 1:
        return [alt]
    raise ValueError(
        f"Numeric attribution target must be single-token, "
        f"got {raw_label!r}->{ids_raw}, alt {alt!r}->{ids_alt}"
    )

all_probe = []
with open(PROBE_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        all_probe.append(json.loads(line))

print(f"All probe prompts: {len(all_probe)}")
print("  by task_type:", dict(Counter(p.get("task_type", "binary") for p in all_probe)))

if RUN_FULL:
    probe_prompts = all_probe
    print(f"RUN_FULL=True: using all {len(probe_prompts)} prompts.")
else:
    strata: dict = defaultdict(list)
    for p in all_probe:
        key = (p.get("family", "?"), p.get("tail", "?"), str(p.get("label", "?")))
        strata[key].append(p)

    rng = random.Random(42)
    pilot = []
    per_stratum = max(1, N_PILOT // len(strata))
    for key, bucket in sorted(strata.items()):
        rng.shuffle(bucket)
        pilot.extend(bucket[:per_stratum])

    remaining_pool = [p for p in all_probe if p not in pilot]
    rng.shuffle(remaining_pool)
    while len(pilot) < N_PILOT and remaining_pool:
        pilot.append(remaining_pool.pop(0))

    probe_prompts = pilot[:N_PILOT]
    dist = Counter(
        (p.get("family", "?"), p.get("tail", "?"), str(p.get("label", "?")))
        for p in probe_prompts
    )
    print(f"Pilot subset: {len(probe_prompts)} prompts")
    for key, cnt in sorted(dist.items()):
        print(f"  {key}: {cnt}")

Binary probe prompts: 270
RUN_FULL=True: running all 270 binary prompts.


## 4 — First-token accuracy (post-LoRA), binary + numeric

In [5]:
overall_correct = 0

binary_correct = 0
binary_total = 0
correct_true = 0
total_true = 0
correct_false = 0
total_false = 0

numeric_correct = 0
numeric_total = 0

predictions = []

with torch.inference_mode():
    for entry in probe_prompts:
        logits, _ = ct_model.feature_intervention(entry["prompt"], [])
        last_logit = logits.squeeze(0)[-1, :]
        pred_id = int(last_logit.argmax().item())
        pred_tok = ct_tokenizer.decode([pred_id])

        task_type = entry.get("task_type", "binary")
        if task_type == "binary":
            label_id = VOCAB_ID_TRUE if entry["label"] else VOCAB_ID_FALSE
            binary_total += 1
            is_ok = pred_id == label_id
            if is_ok:
                binary_correct += 1
                overall_correct += 1
            if entry["label"]:
                total_true += 1
                if pred_id == VOCAB_ID_TRUE:
                    correct_true += 1
            else:
                total_false += 1
                if pred_id == VOCAB_ID_FALSE:
                    correct_false += 1
        else:
            raw_label = entry["label_token"]
            label_ids = ct_tokenizer.encode(raw_label, add_special_tokens=False)
            if len(label_ids) != 1:
                alt = raw_label.lstrip()
                alt_ids = ct_tokenizer.encode(alt, add_special_tokens=False)
                if len(alt_ids) == 1:
                    label_ids = alt_ids
                else:
                    raise ValueError(
                        f"Numeric label_token must encode to single id, "
                        f"got {raw_label!r}->{label_ids}, alt {alt!r}->{alt_ids}"
                    )
            label_id = int(label_ids[0])
            numeric_total += 1
            is_ok = pred_id == label_id
            if is_ok:
                numeric_correct += 1
                overall_correct += 1

        predictions.append({
            "prompt_id": entry["prompt_id"],
            "task_type": task_type,
            "pred_token": pred_tok,
            "pred_id": pred_id,
            "is_correct": is_ok,
        })

n = len(probe_prompts)
print(f"Post-LoRA first-token accuracy (n={n}):")
print(f"  Overall : {overall_correct}/{n} = {overall_correct/n:.1%}")
if binary_total:
    print(f"  Binary  : {binary_correct}/{binary_total} = {binary_correct/binary_total:.1%}")
    if total_true:
        print(f"    True : {correct_true}/{total_true} = {correct_true/total_true:.1%}")
    if total_false:
        print(f"    False: {correct_false}/{total_false} = {correct_false/total_false:.1%}")
if numeric_total:
    print(f"  Numeric : {numeric_correct}/{numeric_total} = {numeric_correct/numeric_total:.1%}")
print("(Baseline notebook `02` uses the same split for comparison.)")

Post-LoRA first-token accuracy (n=40):
  Overall : 20/40 = 50.0%
  True    : 11/18 = 61.1%
  False   : 9/22 = 40.9%

Baseline first-token accuracy was ~18.5% on binary prompts.
An increase here confirms the LoRA changed the output distribution.


## 5 — Attribution + stats loop

Same **`compute_statistics`** schema as **`02`**. Compare in `05_lora_comparison.ipynb` on **`prompt_id`**.

**Checkpoint / resume**

- Any row with **`attribution_succeeded`** already in **`STATS_FILE`** is skipped (no rerun).

- **`RUN_FULL=False`** only appends to **`stats_lora_combined_pilot.json`**.

- **`RUN_FULL=True`** appends to **`stats_lora_combined.json`**. Prompts that succeeded in **`stats_lora_combined_pilot.json`** but are still missing from the full file have their rows **copied** over (instant, no GPU) so you avoid duplicate attribution **and** a completed full run fills **`stats_lora_combined.json`** for **`USE_PILOT=False`** in `05`.

In [6]:
import importlib
import utils.graph_statistics as gs_mod
importlib.reload(gs_mod)

from circuit_tracer import attribute
from utils.graph_statistics import compute_statistics, load_statistics, append_statistic

def _successful_rows_by_id(rows):
    return {
        e["prompt_id"]: e
        for e in rows
        if e.get("prompt_id") is not None and e.get("attribution_succeeded")
    }

existing = load_statistics(STATS_FILE)
done_rows = _successful_rows_by_id(existing)
done_ids = set(done_rows)

pilot_succ: dict = {}
if RUN_FULL and STATS_FILE.resolve() != PILOT_STATS_FILE.resolve():
    pilot_succ = _successful_rows_by_id(load_statistics(PILOT_STATS_FILE))

n_prev_failed = sum(1 for e in existing if not e.get("attribution_succeeded"))
remaining = [p for p in probe_prompts if p["prompt_id"] not in done_ids]
n_copy_hint = sum(1 for p in remaining if p["prompt_id"] in pilot_succ)

print(f"Stats file (append target) : {STATS_FILE}")
print(f"Already succeeded (here) : {len(done_ids)}")
if RUN_FULL and pilot_succ:
    print(f"Pilot rows to reuse      : {n_copy_hint}")
print(f"Previously failed (here) : {n_prev_failed} (eligible for retry)")
print(f"Remaining to process     : {len(remaining)} prompts")
print()

t0 = time.time()
n_success = 0
n_fail = 0
n_copied = 0
progress_every = 5

for i, entry in enumerate(remaining, start=1):
    pid = entry["prompt_id"]
    task_type = entry.get("task_type", "binary")
    print(f"[{i}/{len(remaining)}] {pid} ({task_type}) ...", end=" ", flush=True)

    if RUN_FULL and pid in pilot_succ:
        append_statistic(dict(pilot_succ[pid]), STATS_FILE)
        n_success += 1
        n_copied += 1
        print("OK (reused pilot row)")
        continue

    try:
        graph = attribute(
            prompt=entry["prompt"],
            model=ct_model,
            attribution_targets=_resolve_attr_targets(entry),
            **ATTR_KWARGS,
        )
        stat = compute_statistics(graph, entry, phase=PHASE)
        append_statistic(stat, STATS_FILE)
        n_success += 1

        del graph
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        density  = stat.get("edge_density", float("nan"))
        n_active = stat.get("n_active_features", "?")
        print(f"OK  n_active={n_active}  density={density:.4f}")

    except Exception as exc:
        n_fail += 1
        stat = {
            "prompt_id": pid,
            "phase": PHASE,
            "task_type": task_type,
            "family": entry.get("family"),
            "tail": entry.get("tail"),
            "label": entry["label"],
            "label_token": entry["label_token"],
            "template_id": entry.get("template_id"),
            "attribution_succeeded": False,
            "_error": str(exc),
        }
        append_statistic(stat, STATS_FILE)
        print(f"FAIL: {exc}")
        traceback.print_exc()

    if i % progress_every == 0:
        elapsed = time.time() - t0
        speed   = i / elapsed
        eta     = (len(remaining) - i) / speed if speed > 0 else float("inf")
        print(
            f"[progress] {i}/{len(remaining)} ({i/len(remaining):.0%}) | "
            f"success={n_success}/{n_success+n_fail} | "
            f"{speed:.3f} prompts/s | ETA {eta/60:.1f} min"
        )

print()
print(f"Done. {n_success} ok ({n_copied} reused from pilot), {n_fail} failed in {time.time()-t0:.0f}s.")
print(f"Stats saved to: {STATS_FILE}")

Stats file (append target) : /workspace/thesis_circuit_breaker/my_work/results/statistics/stats_lora_combined_pilot.json
Already succeeded (here) : 0
Previously failed (here) : 0 (eligible for retry)
Remaining to process     : 40 prompts

[1/40] tri_v2_167 ... OK  n_active=23659  density=0.1160
[2/40] tri_v2_029 ... 

KeyboardInterrupt: 

## 6 — Scale-up decision

After the pilot: if **`n_success >= N_PILOT * 0.9`** (or whenever you want the full probe set),

set **`RUN_FULL = True`** in §1 and **re-run from §1**.

§5 skips IDs already recorded in **`stats_lora_combined.json`** (if any) **and**

**reuses pilot successes by copying rows** — no duplicate attribution cost for prompts that finished during the pilot.

In [ ]:
final_stats = load_statistics(STATS_FILE)
n_ok    = sum(1 for s in final_stats if s.get("attribution_succeeded"))
n_total = len(final_stats)

print(f"=== Post-run summary ===")
print(f"Stats file : {STATS_FILE}")
print(f"Total rows : {n_total}")
print(f"Succeeded  : {n_ok}  ({n_ok/max(n_total,1):.1%})")
print()
if not RUN_FULL:
    print("Pilot complete.")
    print("If results look good: set RUN_FULL=True in §1 and re-run from §1 onwards.")
    print("Full run saves to:", FULL_STATS_FILE)
else:
    print("Full run complete. Proceed to 05_lora_comparison.ipynb.")